# 第10周 · CNN 与归纳偏置（RNN 概念）

**本周结束时你能做到**：手算二维卷积，并用 $\lfloor (W-K+2P)/S \rfloor + 1$ 预测输出尺寸；解释**局部性**与**平移等变性**这两种归纳偏置，并用"参数量 vs 准确率"的数字说明它们的价值；照着 PyTorch 官方教程在 CIFAR-10 上训练你的第一个彩色图像 CNN；用误差分析和"训练 / 训练-开发 / 目标开发 / 人类水平"表判断下一步该修什么，并用 CIFAR-10-C 式的腐蚀扰动测一次分布偏移；把卷积搬到文本上搭出 TextCNN，并**诚实地**与第 6 周的 TF-IDF 基线比较；说清 RNN 的隐藏状态与梯度消失，以及它为什么被注意力取代。

本周的蓝本：

- 图像 CNN：PyTorch 官方教程 [Training a Classifier](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)（CIFAR-10；网络与训练配置照搬）；
- TextCNN：Kim (2014) [Convolutional Neural Networks for Sentence Classification](https://arxiv.org/abs/1408.5882)（多宽度卷积核 + max-over-time pooling + dropout 0.5），代码结构参照 [bentrevett/pytorch-sentiment-analysis](https://github.com/bentrevett/pytorch-sentiment-analysis) 的第 3 个教程（MIT 许可）；
- Sobel 边缘检测核：[Wikipedia: Sobel operator](https://en.wikipedia.org/wiki/Sobel_operator)；
- 分布偏移：Hendrycks & Dietterich (2019) [Benchmarking Neural Network Robustness to Common Corruptions and Perturbations](https://arxiv.org/abs/1903.12261) 定义的 CIFAR-10-C 腐蚀，代码改编自 [hendrycks/robustness](https://github.com/hendrycks/robustness)（Apache-2.0 许可，工具包 `zero2researcher.shift`）。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms
from zero2researcher import data, text, experiments
from zero2researcher.checks import week10 as ck

# 课程统一的 device 选择：有 CUDA 用 CUDA（Colab GPU），Apple 芯片用 MPS，否则 CPU
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

---
## S1 · 卷积：一个在图上滑动的小窗口

第 8 周的 MLP 先把 28×28 的图**展平**成 784 个数，第一层的每个神经元都和全部 784 个像素相连。它并不知道哪两个像素是邻居——把所有图片的像素按同一种方式打乱顺序再训练，MLP 的成绩几乎不变。

**卷积**换了一种做法：拿一个很小的权重矩阵（**卷积核**，例如 3×3），在图上从左到右、从上到下滑动；每到一处，把核与它盖住的那一小块图**逐元素相乘再求和**，得到输出图上的一个数：

$$\text{out}[i, j] = \sum_{u=0}^{K-1} \sum_{v=0}^{K-1} \text{img}[i+u,\ j+v] \cdot \text{kernel}[u, v]$$

### 先在纸上算一个数

一张 5×5 的小图（正中一条竖直亮线），一个 3×3 的核。**先在纸上**算出输出左上角那个数，再运行验证：

In [ ]:
img5 = np.array([[0, 0, 9, 0, 0],
                 [0, 0, 9, 0, 0],
                 [0, 0, 9, 0, 0],
                 [0, 0, 9, 0, 0],
                 [0, 0, 9, 0, 0]], dtype=float)
kernel = np.array([[-1, 0, 1],
                   [-1, 0, 1],
                   [-1, 0, 1]], dtype=float)     # 右边亮于左边 → 正；左边亮于右边 → 负

# 在这里写你的代码
raise NotImplementedError
print(patch)
print("out[0, 0] =", out00)        # 每行 (-1)·0 + 0·0 + 1·9 = 9，三行合计 27

核滑到每一个位置都做同样的事——两层循环就够了：

In [ ]:
def conv2d_valid(img, kernel):
    """valid 卷积（不补边、步长 1）：核只停在完全落在图内的位置"""
    kh, kw = kernel.shape
    oh, ow = img.shape[0] - kh + 1, img.shape[1] - kw + 1
    out = np.zeros((oh, ow))
    # 在这里写你的代码
    raise NotImplementedError
    return out


print(conv2d_valid(img5, kernel))

亮线的左边缘处输出 $+27$、右边缘处 $-27$、正中间 $0$：这个核是一个**竖直边缘检测器**——它只对"左右亮度不同"的地方有反应。

### 输出尺寸：valid、padding、stride

5×5 的图配 3×3 的核，输出是 3×3——核只有 3×3 个位置能完整落在图内。两个旋钮改变输出尺寸：

- **padding $P$**：在图四周补 $P$ 圈 0，让核能"探出"边界；3×3 的核配 $P=1$，输出与输入一样大；
- **stride $S$**：核每次跳 $S$ 格；$S=2$ 时输出边长约减半。

$$W_{\text{out}} = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1$$

（$2P$ 是因为两边都补；除不尽时向下取整——最后那段放不下一个完整的核，就丢掉。）用 `nn.Conv2d` 验证这个公式——只看形状，不训练：

In [ ]:
x = torch.zeros(1, 1, 28, 28)          # (batch, 通道, 高, 宽)：一张 MNIST 尺寸的灰度图
for K, P, S in [(3, 0, 1), (3, 1, 1), (5, 0, 1), (3, 1, 2), (5, 2, 2)]:
    conv = nn.Conv2d(1, 1, kernel_size=K, padding=P, stride=S)
    formula = (28 - K + 2 * P) // S + 1
    print(f"K={K} P={P} S={S}:  nn.Conv2d 输出 {tuple(conv(x).shape[-2:])}   公式 {formula}")

### Sobel：真实图片上的边缘检测

Irwin Sobel 与 Gary Feldman 1968 年提出的 Sobel 算子（[Wikipedia](https://en.wikipedia.org/wiki/Sobel_operator)）就是两个 3×3 的核：$G_x$ 对左右方向的亮度变化敏感（竖直边缘），$G_y$ 对上下方向的变化敏感（水平边缘）：

$$G_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix} \qquad G_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix}$$

（对照资料时注意：Sobel 核的正负号在不同资料里约定不一。当前版本的 Wikipedia 写出的 $G_x$、$G_y$ 与这里完全相同，但它把公式里的 $*$ 定义为信号处理的真卷积——先把核翻转 180° 再滑动；PyTorch 做的是不翻转的互相关。同一个矩阵用这两种运算作用在同一张图上，得到的边缘图正好差一个符号。练习 E2 会讲清两者的关系。）

把它们作用在一张 MNIST 数字上。MNIST 下载到课程统一的缓存目录 `data.download_root()`（约 11MB），之后各周共用：

In [ ]:
tfm_mnist = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.1307,), (0.3081,))])
mnist_train = datasets.MNIST(root=data.download_root(), train=True, download=True, transform=tfm_mnist)
mnist_test = datasets.MNIST(root=data.download_root(), train=False, download=True, transform=tfm_mnist)

digit = mnist_train.data[0].numpy() / 255.0       # 第一张训练图（一个 5），0–1 灰度，28×28
Gx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)
Gy = Gx.T                                          # 转置：对上下方向的变化敏感
# 在这里写你的代码
raise NotImplementedError

fig, axes = plt.subplots(1, 4, figsize=(11, 3.2), layout="constrained")
panels = [(digit, "Input digit (28x28)", "gray"), (ex, "Gx: vertical edges", "RdBu_r"),
          (ey, "Gy: horizontal edges", "RdBu_r"), (magnitude, "Edge magnitude (26x26)", "gray")]
for ax, (im, title, cmap) in zip(axes, panels):
    v = np.abs(im).max()
    ax.imshow(im, cmap=cmap, vmin=-v if cmap == "RdBu_r" else 0, vmax=v)
    ax.set_title(title)
    ax.axis("off")
fig.suptitle("Two hand-designed 3x3 kernels already outline the digit")
plt.show()

### 两个归纳偏置

一个 3×3 的核只有 9 个权重，却被**同一套**权重扫遍整张图。这背后是对图像数据的两条假设——它们叫**归纳偏置**（inductive bias）：模型在看到任何数据之前，就已经"相信"的东西。

1. **局部性**（locality）：有用的模式（笔画、边缘、角点）由**相邻**的像素组成——所以每个输出只看一小块；
2. **平移等变性**（translation equivariance）：同一个模式出现在图的哪里都是同一个模式——所以全图**共享**一个核。输入平移多少，输出就跟着平移多少。

第 2 条可以直接验证：把数字右移 3 个像素再做 Sobel，结果应该就是原来的 Sobel 结果右移 3 格：

In [ ]:
a = conv2d_valid(digit, Gx)                           # 原图的边缘图
b = conv2d_valid(np.roll(digit, 3, axis=1), Gx)       # 右移 3 像素后的边缘图（数字两侧都是 0，不会绕出边界）
print("b 的第 3 列起 vs a 的前 23 列，最大差异:", np.abs(b[:, 3:] - a[:, :-3]).max())

差异恰好是 0：**先平移再卷积 = 先卷积再平移**。

> **类比**：架构对数据的假设，就像实验设计里内置的前提。做细胞计数时，你默认"一个细胞不论出现在视野左上还是右下，都用同一个标准判定"——这条前提让你只需要一套判定规则，而不必为每个位置单独定标准。前提**成立**时，它让你用更少的样本得到更可靠的结论；前提**不成立**时（比如视野边缘有系统性的光照偏差），它会悄悄把偏差带进每一个结果。CNN 的两条前提对自然图像大体成立，所以它能省下大量参数——下面用数字说话。

### 池化：把"在哪儿"变得模糊一点

**最大池化**（max pooling）把每个 2×2 的小块缩成一个数（取最大值），步长 2 时宽高减半。它没有可学习的参数，作用有二：缩小特征图、减少后续计算；以及让输出对**小幅平移**不那么敏感——2×2 块里的最大值无论落在块内哪个角，池化结果都一样。**先在纸上写出答案**，再运行：

In [ ]:
m = torch.tensor([[1., 3., 0., 2.],
                  [4., 2., 1., 0.],
                  [0., 1., 5., 6.],
                  [2., 2., 7., 1.]])
pool = nn.MaxPool2d(kernel_size=2, stride=2)
print(pool(m[None, None])[0, 0])        # nn.MaxPool2d 要 (N, C, H, W)：用 [None, None] 补出两个维度

### MNIST：CNN vs MLP，比参数量，也比准确率

两个模型在**同一份**训练子集（前 10,000 张）、同一优化器（Adam 1e-3）、同样 3 个 epoch 下比较，测试集为全部 10,000 张：

- **MLP**：第 8 周的 784 → 256 → 10；
- **小 CNN**：两层 3×3 卷积（通道数与核大小同 [pytorch/examples 的 MNIST 示例](https://github.com/pytorch/examples/tree/main/mnist)：1→32→64、3×3。不同的是我们每层都补一圈边（`padding=1`）、每层后都池化，也不要它的 dropout 和 9216→128 的大全连接层），每层后接 ReLU 与 2×2 最大池化，最后一个全连接层直接输出 10 类。

先写出 CNN 并逐层打印形状——**在每一行旁边注释 shape，是本周起的硬性习惯**：

In [ ]:
class MLP(nn.Module):                      # 第 8 周的原样
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10))

    def forward(self, x):
        return self.net(x)


class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # 在这里写你的代码
        raise NotImplementedError

    def forward(self, x):
        # 在这里写你的代码
        raise NotImplementedError


def n_params(model):
    return sum(p.numel() for p in model.parameters())


x = torch.zeros(2, 1, 28, 28)
cnn = SmallCNN()
for layer in cnn.features:
    x = layer(x)
    print(f"{layer.__class__.__name__:10s} → {tuple(x.shape)}")
print(f"\n参数量  MLP: {n_params(MLP()):,}   SmallCNN: {n_params(cnn):,}")

CNN 的参数只有 MLP 的四分之一。逐层看看参数花在了哪里：

In [ ]:
for name, p in cnn.named_parameters():
    print(f"{name:22s} {str(tuple(p.shape)):18s} {p.numel():>7,}")

两个卷积层合计不到 2 万个参数——一个核的权重被全图共享，图再大，核也不变大。参数的大头反而在最后的全连接层。

训练与评估用第 8 周你写过的两个函数（原样搬来）：

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """训练一个 epoch，返回平均训练损失。"""
    model.train()
    total, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = criterion(model(xb), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
        n += len(xb)
    return total / n


def evaluate(model, loader, device):
    """返回准确率；eval 模式 + 全程 no_grad，绝不改模型。"""
    model.eval()
    correct, n = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            correct += (model(xb).argmax(dim=1) == yb).sum().item()
            n += len(yb)
    return correct / n

In [ ]:
mnist_loader = DataLoader(Subset(mnist_train, range(10000)), batch_size=64, shuffle=True)
mnist_test_loader = DataLoader(mnist_test, batch_size=1000)

mnist_results = {}
for Model in (MLP, SmallCNN):
    experiments.set_seed(42)
    model = Model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    t0 = time.time()
    for epoch in range(3):
        train_one_epoch(model, mnist_loader, nn.CrossEntropyLoss(), optimizer, device)
    acc = evaluate(model, mnist_test_loader, device)
    mnist_results[Model.__name__] = (n_params(model), acc)
    print(f"{Model.__name__:9s} 参数 {n_params(model):>8,}   测试准确率 {acc:.4f}   用时 {time.time() - t0:.1f}s")

ck.mnist_cnn_accuracy(mnist_results["SmallCNN"][1])

In [ ]:
names = list(mnist_results)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), layout="constrained")
bars = axes[0].bar(names, [mnist_results[n][0] for n in names], color=["C0", "C2"])
axes[0].bar_label(bars, fmt="{:,.0f}")
axes[0].set_ylim(0, 1.15 * max(mnist_results[n][0] for n in names))
axes[0].set_ylabel("Trainable parameters")
axes[0].set_title("Parameters")
bars = axes[1].bar(names, [mnist_results[n][1] for n in names], color=["C0", "C2"])
axes[1].bar_label(bars, fmt="%.3f")
axes[1].set_ylim(0.9, 1.0)
axes[1].set_ylabel("Test accuracy")
axes[1].set_title("Accuracy (10k training images, 3 epochs)")
fig.suptitle("The CNN uses 4x fewer parameters and makes less than half the errors")
plt.show()

参数少 4 倍，测试错误率却从约 6% 降到约 2.5%。这就是"对的归纳偏置"的价值：它把"像素有邻居、模式可以出现在任何位置"这两条知识直接写进了架构，模型不必从 1 万张图里自己悟出来。

---
## S2 · CIFAR-10：照着官方教程训练第一个彩色图像 CNN

[CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html)（Krizhevsky, 2009）：60,000 张 32×32 的**彩色**小图，10 类各 6,000 张，划分为 50,000 训练 / 10,000 测试。和 MNIST 比，物体有不同的姿态、背景和光照，难得多。

本节**照搬** PyTorch 官方教程 [Training a Classifier](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)：归一化方式、网络结构、损失、优化器、batch 大小和 epoch 数都与官方一致。这样你的结果可以直接和官方报告的数字对照——这本身就是一次**阳性对照**：先复现别人的已知结果，再在它上面做改动。

首次运行会下载约 163MB（之后从缓存 `data.download_root()` 读取，第一次可能要等几分钟）：

In [ ]:
tfm_cifar = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])   # 官方：每个通道映射到 [-1, 1]
cifar_train = datasets.CIFAR10(root=data.download_root(), train=True, download=True, transform=tfm_cifar)
cifar_test = datasets.CIFAR10(root=data.download_root(), train=False, download=True, transform=tfm_cifar)
classes = cifar_train.classes
print(len(cifar_train), len(cifar_test), classes)
print("一张图的形状:", tuple(cifar_train[0][0].shape), "  ← (通道 3, 高 32, 宽 32)")

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    img, label = cifar_train[i]
    ax.imshow((img.permute(1, 2, 0) * 0.5 + 0.5).numpy())    # (C, H, W) → (H, W, C)，并撤销归一化
    ax.set_title(classes[label])
    ax.axis("off")
fig.suptitle("CIFAR-10: tiny colour photos with varied poses and backgrounds")
plt.show()

### 网络：官方教程的 `Net`

```
输入 (3, 32, 32) → Conv 5×5, 6 个核 → ReLU → MaxPool 2 → Conv 5×5, 16 个核 → ReLU → MaxPool 2
                → 展平 16×5×5 = 400 → 全连接 120 → 84 → 10
```

彩色图有 3 个输入通道，所以第一层的每个核是 5×5×**3**：它同时看一小块区域的红、绿、蓝。用尺寸公式自己验证 400 这个数：32 → 28 → 14 → 10 → 5。

In [ ]:
class Net(nn.Module):                          # 照搬官方教程
    def __init__(self):
        super().__init__()
        # 在这里写你的代码
        raise NotImplementedError

    def forward(self, x):
        # 在这里写你的代码
        raise NotImplementedError


print("官方 Net 参数量:", f"{n_params(Net()):,}")

### 训练前先体检：初始损失

第 9 周诊断配方的第一条：随机初始化的 10 类分类器，初始交叉熵应约等于 $\ln 10 \approx 2.303$（模型对 10 类"一视同仁"）。差得远，就先别训练，查损失函数和数据。

In [ ]:
cifar_test_loader = DataLoader(cifar_test, batch_size=1000)
experiments.set_seed(42)
net = Net().to(device)
criterion = nn.CrossEntropyLoss()
with torch.no_grad():
    xb, yb = next(iter(cifar_test_loader))
    init_loss = criterion(net(xb.to(device)), yb.to(device)).item()
print(f"初始损失 {init_loss:.4f}   ln(10) = {np.log(10):.4f}")
ck.initial_loss(init_loss)

### 训练：官方配置

`batch_size=4`、`SGD(lr=0.001, momentum=0.9)`、2 个 epoch，每 2,000 个 mini-batch 打印一次平均损失——与官方完全相同。CPU 上约 1 分钟。网络很小、batch 只有 4，GPU 的加速并不明显（官方教程原话："Because your network is really small"）。

In [ ]:
N_TRAIN = 50000          # 机器太慢时可改成 10000，并把 n_train=10000 传给下面的自查
cifar_loader = DataLoader(Subset(cifar_train, range(N_TRAIN)), batch_size=4, shuffle=True)
optimizer = torch.optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

t0 = time.time()
for epoch in range(2):
    net.train()
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(cifar_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        # 在这里写你的代码
        raise NotImplementedError
        running_loss += loss.item()
        if i % 2000 == 1999:
            print(f"[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}")
            running_loss = 0.0
print(f"训练用时 {time.time() - t0:.0f}s")

blitz_acc = evaluate(net, cifar_test_loader, device)
print(f"测试准确率: {blitz_acc:.4f}   （官方页报告 55%；随机猜是 10%）")
ck.cifar_blitz_accuracy(blitz_acc, n_train=N_TRAIN)

官方教程接着看了**逐类**准确率——总体 55% 背后，各类差别很大：

In [ ]:
correct, total = torch.zeros(10), torch.zeros(10)
net.eval()
with torch.no_grad():
    for xb, yb in cifar_test_loader:
        pred = net(xb.to(device)).argmax(dim=1).cpu()
        for c in range(10):
            mask = yb == c
            correct[c] += (pred[mask] == c).sum()
            total[c] += mask.sum()
per_class = (correct / total).numpy()
order = np.argsort(per_class)

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.barh([classes[i] for i in order], per_class[order], color="C0")
ax.axvline(0.1, color="gray", ls="--", label="Chance (0.10)")
ax.set_xlabel("Test accuracy")
ax.set_title("Per-class accuracy varies widely around the 55% average")
ax.legend(loc="lower right")
plt.show()

### 与 MLP 对比：同样的训练配置，6 倍的参数

把卷积部分拿掉，只留官方的全连接头（3072 → 120 → 84 → 10），其余训练配置一字不改：

In [ ]:
experiments.set_seed(42)
mlp_cifar = nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 120), nn.ReLU(),
                          nn.Linear(120, 84), nn.ReLU(), nn.Linear(84, 10)).to(device)
optimizer = torch.optim.SGD(mlp_cifar.parameters(), lr=0.001, momentum=0.9)
t0 = time.time()
for epoch in range(2):
    train_one_epoch(mlp_cifar, cifar_loader, criterion, optimizer, device)
mlp_cifar_acc = evaluate(mlp_cifar, cifar_test_loader, device)
print(f"MLP 参数 {n_params(mlp_cifar):>8,}   测试准确率 {mlp_cifar_acc:.4f}   用时 {time.time() - t0:.0f}s")
print(f"CNN 参数 {n_params(net):>8,}   测试准确率 {blitz_acc:.4f}")

MLP 的参数是 CNN 的 6 倍，准确率却低了约 6 个百分点。和 MNIST 上的结论一致，而且在更难的彩色图上更明显。

### 数据增强：让模型见到"同一张图的不同样子"

50,000 张训练图对 10 类自然物体并不多，模型容易记住训练集。**数据增强**在每次取样时对图片做随机的、**不改变类别**的变换：

- `RandomCrop(32, padding=4)`：四周补 4 像素后随机裁回 32×32——相当于随机平移最多 4 个像素；
- `RandomHorizontalFlip()`：一半概率左右翻转——镜子里的猫还是猫。

增强也是一种先验：你在告诉模型"这些变换不改变答案"。（上下翻转就不行：倒立的轮船在 CIFAR 里几乎不存在。）增强只加在**训练集**上，测试集保持原样。

In [ ]:
from PIL import Image

tfm_aug = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

experiments.set_seed(0)
pil = Image.fromarray(cifar_train.data[7])        # .data 是原始 uint8 数组 (50000, 32, 32, 3)
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
axes[0].imshow(pil)
axes[0].set_title("Original")
for ax in axes[1:]:
    ax.imshow((tfm_aug(pil).permute(1, 2, 0) * 0.5 + 0.5).numpy())
    ax.set_title("Augmented")
for ax in axes:
    ax.axis("off")
fig.suptitle("Random crop + flip: same label, slightly different pixels every epoch")
plt.show()

每一张都还是那匹马，只是位置挪了几个像素、或者左右镜像了（黑边就是 padding 补的 0）。用法只有一处改动：训练集的 `transform` 换成 `tfm_aug`（`datasets.CIFAR10(..., train=True, transform=tfm_aug)`），训练循环一行不改。增强的好处通常要更多 epoch 才显现——这里的 2 个 epoch 看不出来，有 GPU 的话可以选做（加增强、训 15 个 epoch，看测试准确率能涨多少）。

### 打开黑箱：第一层的 6 个卷积核

`net.conv1.weight` 的形状是 `(6, 3, 5, 5)`：6 个核，每个都是 5×5×3（红、绿、蓝各一片）。把每个核单独缩放到 0–1，当作一张 5×5 的小彩图画出来：

In [ ]:
w = net.conv1.weight.detach().cpu()               # (6, 3, 5, 5)
print("第一层卷积核的形状:", tuple(w.shape))
fig, axes = plt.subplots(1, 6, figsize=(9, 1.9))
for i, ax in enumerate(axes):
    k = w[i]
    k = (k - k.min()) / (k.max() - k.min())        # 每个核单独缩放到 [0, 1]
    ax.imshow(k.permute(1, 2, 0).numpy())          # (3, 5, 5) → (5, 5, 3)：当作 RGB 小图
    ax.set_title(f"Kernel {i}")
    ax.axis("off")
fig.suptitle("First-layer kernels: coarse colour blobs and light/dark contrasts")
plt.show()

只训练了 2 个 epoch、只有 6 个核，图案还很粗糙，但已经能看出两类东西：有的核偏向某种颜色（颜色检测器），有的核一侧亮一侧暗（明暗/边缘检测器）。更大的网络训练充分后，第一层几乎总会长出方向分明的边缘检测器和颜色斑块——和 S1 里手工设计的 Sobel 核是同一类东西，只不过这次是**从数据里学出来的**。

### 研究方法插曲：误差分析与数据错配

模型训好了，测试准确率也对上了官方的 55%。下一步该做什么——加深网络？加数据？加增强？吴恩达在 [CS230](https://cs230.stanford.edu/) 和《Machine Learning Yearning》里给这一步定了两个固定动作：**看错例**（误差分析），以及**把误差拆成几块**（偏差 / 方差 / 数据错配）。两者都要用训练集以外、专门留出来的数据，而且要分清哪一份可以反复看错例、哪一份只能算总误差。

**先改一条规矩**。前面那张逐类准确率图是在全部 1 万张测试图上算的——当时只是看看，没据此做任何决定，问题不大。可是从现在起要根据结果决定下一步了；如果继续在测试集上看错例、挑方法，测试集就不再是"没见过的数据"，第 5 周讲过的泄漏就回来了。所以把官方测试集一分为二：

- **前 5000 张 = 训练-开发集**（train-dev set）：干净、与训练集同分布，看错例用——随便看、据此做决定；
- **后 5000 张 + 对比度扰动 = 目标开发集**（B 实验室）：等一下会把后 5000 张加上对比度扰动，当作"B 实验室的显微镜"拍出来的数据，也就是模型将来真正要用的分布。规矩是：**不看错例、不调参数，只算总误差**（也就是后面误差表里"目标开发"那一行）。

真实项目里还要从目标分布再留一份**没碰过的测试集**，最后在它上面报告校正后的成绩；这里为了简单，没有再拆分。

下面直接用刚训练好的 `net`，不用重训。先写两个小工具（给出）：把原始 uint8 图像变成与 `tfm_cifar` 相同的输入，以及批量预测：

In [ ]:
from zero2researcher import shift          # 本插曲后半用到的图像腐蚀函数（见下文"分布偏移"）


def to_input(u8):
    # uint8 图像 (N, 32, 32, 3) → 与 tfm_cifar 相同的输入 (N, 3, 32, 32)：ToTensor，再 Normalize(0.5, 0.5)
    return (torch.from_numpy(u8).float().div(255).permute(0, 3, 1, 2) - 0.5) / 0.5


def predict(model, u8, batch_size=1000):
    # 对一批 uint8 图像做预测，返回 NumPy 数组；eval 模式 + no_grad
    model.eval()
    with torch.no_grad():
        return torch.cat([model(to_input(u8[i:i + batch_size]).to(device)).argmax(dim=1).cpu()
                          for i in range(0, len(u8), batch_size)]).numpy()


test_u8, test_y = cifar_test.data, np.array(cifar_test.targets)    # .data 是原始 uint8 数组 (10000, 32, 32, 3)
train_dev_u8, train_dev_y = test_u8[:5000], test_y[:5000]          # 训练-开发集：干净、同分布，看错例用
held_u8, held_y = test_u8[5000:], test_y[5000:]                    # 后 5000 张：加上对比度扰动就是目标开发集，只算总误差
train_dev_pred = predict(net, train_dev_u8)
train_dev_error = float((train_dev_pred != train_dev_y).mean())
print(f"训练-开发集误差 {train_dev_error:.3f}，共 {(train_dev_pred != train_dev_y).sum()} 个错例")

#### 第一步：看 100 个错例

两千多个错例看不完，抽 100 个就够找出占比在 5% 以上的大类。找出训练-开发集里所有预测错的下标，再用 `np.random.default_rng(0)` **无放回**地抽 100 个，存进 `sample`（排一下序，画图时好对照）：

In [ ]:
# 在这里写你的代码
raise NotImplementedError
ck.error_sample(sample, train_dev_y, train_dev_pred)

In [ ]:
fig, axes = plt.subplots(10, 10, figsize=(14, 16), layout="constrained")
for k, (ax, i) in enumerate(zip(axes.flat, sample)):
    ax.imshow(train_dev_u8[i], interpolation="nearest")
    ax.set_title(f"{k}: {classes[train_dev_y[i]][:5]}→{classes[train_dev_pred[i]][:5]}", fontsize=8)
    ax.axis("off")
fig.suptitle("100 train-dev errors (number: true → predicted)")
plt.show()

现在做误差分析里**只能人工做**的那一步：逐张看这 100 个错例，自己定 4–6 个类别，给每张图打标签。类别从图里来，不要事先定死。常见的有：

- 非典型外观（玩具、绘画、罕见视角；CIFAR 的 bird 类里还有鸵鸟）
- 物体太小、太远、被截断，或是特写
- 背景主导（蓝天里的鸟被认成飞机、草地上的鹿被认成青蛙）
- 太暗或对比度低
- 清晰典型、人一眼就能认出——"模型就是错了"
- 标签本身可疑

一张图可以有多个标签。把结果存成列表 `tags`：`tags[k]` 是上图编号 k 那张图的标签列表，单个标签也写成列表。例如：

```python
tags = [[] for _ in range(100)]
for k in [3, 17, 25, 40]:
    tags[k].append("背景主导")
for k in [3, 8, 61]:
    tags[k].append("暗或低对比")
```

32 × 32 的小图看起来比较粗，30 分钟左右就够了。

In [ ]:
# 在这里写你的代码
raise NotImplementedError
ck.error_tags(tags)

每一类值多少？吴恩达把它叫作**收益上限**（ceiling）：就算把这一类错例全部修好，这份数据上的误差最多也只能降低

$$\text{训练-开发误差} \times \frac{\text{这一类的错例张数}}{\text{抽到的错例张数}}$$

（吴恩达是在开发集上看错例；这里看错例的是训练-开发集，所以乘的是训练-开发误差。）比如误差 44%、100 个错例里 8 个是猫狗互认，那么专门解决猫狗问题最多带来 3.5 个百分点。动手之前先算这笔账，可以避免花一个月去修只占 2% 的问题。写 `ceiling_table(tags, error)`（`error` 是看错例那份数据的误差，这里传入 `train_dev_error`），返回 `{类别: (张数, 收益上限的百分点)}`；一张图有两个标签时，两类各计一次：

In [ ]:
def ceiling_table(tags, error):
    # 在这里写你的代码
    raise NotImplementedError


ck.ceiling_table(ceiling_table)
ceiling = pd.DataFrame(ceiling_table(tags, train_dev_error), index=["张数", "收益上限（百分点）"]).T
print(ceiling.sort_values("张数", ascending=False).round(1))

作者的一次运行（CPU，种子 42）：训练-开发误差 44.1%；100 个错例里动物↔动物 56 个（收益上限约 25 个百分点），最常见的混淆是 bird–deer、cat–dog、automobile–truck。作者自己看图归类时，约三分之一是**清晰典型、人一眼就能认出**的图；鸵鸟、玩具、绘画这类非典型外观约五分之一。

误差分析告诉你"错在哪类图"。而"三分之一的错例人一眼就能认出"在提示另一件事：问题可能不在某一类图，而在模型整体还太弱。判断属于哪类问题，看下面这张表。

#### 第二步：把误差拆成几块

设想这个模型要拿到另一个实验室去用，而 B 实验室的显微镜对比度只有 A 的一半。四个误差排成一张表（吴恩达在 CS230 和《Machine Learning Yearning》里讲的就是这张表）：

| 误差 | 用什么数据算 | 这里用的数据 |
| --- | --- | --- |
| 人类水平 | 人在同类图上的误差，当作"能做到多好"的近似 | 约 6%：Ho-Phuoc 2018 测得人类平均准确率 93.91%（[arXiv:1811.07270](https://arxiv.org/abs/1811.07270)） |
| 训练误差 | 训练过的图 | 训练集前 1 万张 |
| 训练-开发误差 | 没训练过、但与训练集**同分布**的图 | 训练-开发集（干净的前 5000 张） |
| 目标开发误差 | **目标分布**的图 | 目标开发集：后 5000 张，对比度减半（CIFAR-10-C 的"对比度"严重度 2） |

相邻两行之差各有含义：

- **可避免偏差** = 训练 − 人类：连见过的图都分不好——欠拟合，该换更大的模型、训更久；
- **方差** = 训练-开发 − 训练：见过和没见过的同类图差多少——过拟合，该加数据、加正则化；
- **数据错配** = 目标开发 − 训练-开发：换了分布之后掉多少——该让训练数据更像目标数据，或者校正目标数据。

哪一块最大，就先修哪一块。"B 实验室"的图由工具包函数 `shift.corrupt_cifar` 生成（下文细讲）。训练-开发误差上面已经算好（`train_dev_error`）。再算两个误差（0–1 的小数）：`train_error`（训练）、`target_error`（目标开发，即 B 实验室的图），以及三个差 `avoidable_bias`、`variance`、`mismatch`：

In [ ]:
target_u8 = shift.corrupt_cifar(held_u8, "contrast", 2)          # 目标开发集（"B 实验室"）：后 5000 张，对比度减半
train_u8, train_y = cifar_train.data[:10000], np.array(cifar_train.targets[:10000])
HUMAN_ERROR = 1 - 0.9391                                          # Ho-Phuoc 2018

# 在这里写你的代码
raise NotImplementedError
table = {"human": HUMAN_ERROR, "train": train_error, "train_dev": train_dev_error, "dev": target_error,
         "avoidable_bias": avoidable_bias, "variance": variance, "mismatch": mismatch}
print(pd.Series({"人类水平": HUMAN_ERROR, "训练": train_error, "训练-开发": train_dev_error, "目标开发（B 实验室）": target_error},
                name="误差").round(3).to_string())
print(f"\n可避免偏差 {100 * avoidable_bias:.1f} 个百分点 | 方差 {100 * variance:.1f} | 数据错配 {100 * mismatch:.1f}")
ck.mismatch_table(table, n_train=N_TRAIN)

作者的一次运行（CPU，种子 42）：训练 42.1%、训练-开发 44.1%、目标开发 61.7%、人类约 6%，即可避免偏差 36 个百分点、方差 2 个、数据错配 18 个；另外两个种子结论相同。

- **可避免偏差最大**：这个只训了 2 个 epoch 的小网络，连训练过的图都分错四成，首先是**欠拟合**——和刚才"三分之一的错例人一眼就能认出"是同一个信号。
- **方差**约 2 个百分点，刚超过两倍标准误（训练误差在 1 万张上算，标准误约 0.5 个百分点；训练-开发误差在 5000 张上算，约 0.7；两者之差的标准误约 0.9）；另两个种子的方差是 1.8 和 0.1，随种子波动。比起 36 个百分点的可避免偏差，它可以忽略。
- **数据错配**有 18 个百分点：对比度减半这一个变化，就让准确率从约 55% 掉到约 38%。

误差分析告诉你"错在哪类图"，误差表告诉你"属于哪类问题"，两者要一起看。

#### 第三步：分布偏移的压力测试

对比度只是一种可能的变化：B 实验室的图也可能更暗、更模糊、噪声更大。Hendrycks & Dietterich（[ICLR 2019](https://arxiv.org/abs/1903.12261)）把这类常见的退化整理成 CIFAR-10-C：15 种基准腐蚀，外加 4 种留作验证用的腐蚀，每种 5 个严重度。这里挑 4 种（其中高斯模糊属于验证用的那 4 种），按论文代码（[hendrycks/robustness](https://github.com/hendrycks/robustness) 的 `make_cifar_c.py`，Apache-2.0 许可）的公式和严重度常数在内存里生成：

| 腐蚀 | 做什么 | 实验室里的对应情形 |
| --- | --- | --- |
| `gaussian_noise` | 每个像素加高斯噪声 | 曝光不足、增益调得太高 |
| `gaussian_blur` | 高斯模糊 | 轻微离焦 |
| `contrast` | 把每个像素向这张图的均值收缩 | 光源变弱、染色变浅 |
| `brightness` | 在 HSV 空间提高亮度 | 曝光过度 |

工具包函数 `shift.corrupt_cifar(images, name, severity)` 实现了这 4 种（改编自上面的论文代码，改了哪些地方写在函数文档里）。两点说明：

- 论文作者建议用他们发布的成品数据集（[CIFAR-10-C](https://zenodo.org/record/2535967)，CC BY 4.0，约 2.9 GB）做评测，因为在内存里生成的腐蚀不保证与它逐位相同。所以我们的数**只在课程内部可比**，不能和论文里的 CIFAR-10-C 结果直接比较。
- 对比度那一种请自己写一遍——一行 NumPy，却是这个模型的死穴。上游的写法：像素除以 255 变到 0–1，按**每张图、每个通道**求均值 m，算 `(x − m) × c + m`，截到 [0, 1] 后乘 255、转回 uint8（直接截断，不四舍五入）。严重度 1–5 对应 c = 0.75、0.5、0.4、0.3、0.15：

In [ ]:
# 对比度公式与严重度常数来自 hendrycks/robustness 的 make_cifar_c.py（Apache-2.0 许可）：
# https://github.com/hendrycks/robustness/blob/master/ImageNet-C/create_c/make_cifar_c.py
def contrast(u8, c):
    # u8：uint8 (N, 32, 32, 3)；c 越小，对比度越低
    # 在这里写你的代码
    raise NotImplementedError


ck.contrast(contrast)

CORRUPTIONS = {                                            # 名字 → (uint8 图像, 严重度) → uint8 图像
    "gaussian_noise": lambda u8, s: shift.corrupt_cifar(u8, "gaussian_noise", s),
    "gaussian_blur": lambda u8, s: shift.corrupt_cifar(u8, "gaussian_blur", s),
    "contrast": lambda u8, s: contrast(u8, shift.SEVERITY["contrast"][s - 1]),     # 用你写的版本
    "brightness": lambda u8, s: shift.corrupt_cifar(u8, "brightness", s),
}
fig, axes = plt.subplots(4, 6, figsize=(9, 6.6))
for r, (name, fn) in enumerate(CORRUPTIONS.items()):
    axes[r, 0].imshow(train_dev_u8[3])
    axes[r, 0].set_ylabel(name, fontsize=9)
    for s in range(1, 6):
        axes[r, s].imshow(fn(train_dev_u8[3:4], s)[0])
    for c, ax in enumerate(axes[r]):
        ax.set_xticks([])
        ax.set_yticks([])
        if r == 0:
            ax.set_title("clean" if c == 0 else f"severity {c}", fontsize=9)
fig.suptitle("Four CIFAR-10-C corruptions at severities 1-5")
plt.tight_layout()
plt.show()

在训练-开发集上扫一遍 4 种腐蚀 × 5 个严重度（同一批图，只换腐蚀，所以是配对比较）。生成加预测在 CPU 上约 7 秒：

In [ ]:
clean_acc = 1 - train_dev_error
sweep = {name: [float((predict(net, fn(train_dev_u8, s)) == train_dev_y).mean()) for s in range(1, 6)]
         for name, fn in CORRUPTIONS.items()}
for name, accs in sweep.items():
    print(f"{name:15s} 严重度 1–5 的准确率: " + "  ".join(f"{a:.3f}" for a in accs) + f"   （干净 {clean_acc:.3f}）")

fig, ax = plt.subplots(figsize=(6, 3.6))
for name, accs in sweep.items():
    ax.plot(range(0, 6), [clean_acc] + accs, "o-", label=name)
ax.axhline(0.1, color="gray", ls="--", label="Chance (0.10)")
ax.set_xlabel("Severity (0 = clean)")
ax.set_ylabel("Train-dev accuracy")
ax.set_title("Contrast is this model's weak spot; noise barely matters")
ax.legend(fontsize=8)
plt.show()
ck.corruption_sweep(sweep, clean_acc, n_train=N_TRAIN)

作者的一次运行（CPU，种子 42，干净 0.559）：对比度严重度 2 / 3 / 5 时只剩 0.386 / 0.305 / 0.155；高斯噪声到严重度 5 只掉 1 个百分点；模糊和亮度到严重度 5 掉约 10 个百分点。另外两个种子规律相同。

对这个模型来说，对比度是死穴，噪声几乎无害。为什么？一个推测：只训了 2 个 epoch、靠 5×5 卷积加池化的小网络，主要靠低频的颜色和明暗线索做判断；逐像素的噪声被池化平均掉了，而降低对比度正好削弱了明暗线索。这只是推测，要验证还得另做实验。

还有一个细节值得留意：在自然分布里，对比度本来就低的图并没有难很多——作者实测，像素标准差最低四分之一的图，错误率只比最高四分之一高约 4 个百分点；可人为把对比度减半，错误率就涨了 17 个百分点。问题不在"低对比度的图天生难"，而在**超出了训练数据的分布**。

#### 第四步：批次校正——生物学家的老办法

做过 qPCR、Western 或高内涵成像的人都熟悉**批次效应**：同一个样本，换一块板、换一天、换一台仪器，读数整体偏高或偏低。常见的补救是按批次标准化：把每个批次的均值和标准差对齐到参照。

同样的办法可以用在图像上，而且**不用重训**：把 B 实验室这一批图每个通道的均值和标准差，对齐到训练集的通道均值和标准差。写 `match_channels(u8, ref_mean, ref_std)`：在 0–1 尺度上，对**整批图**（不是每张图）求每个通道的均值和标准差，然后算 `(x − 批次均值) / 批次标准差 × 参照标准差 + 参照均值`：

In [ ]:
def match_channels(u8, ref_mean, ref_std):
    # 把一批图每个通道的均值和标准差对齐到 ref_mean、ref_std（0–1 尺度），返回 uint8
    x = u8 / 255.0
    # 在这里写你的代码
    raise NotImplementedError
    return np.uint8(np.clip(x, 0, 1) * 255)


x_ref = train_u8.astype(np.float32) / 255                   # 训练集前 1 万张估计通道统计量（与全部 5 万张差不到 0.002）
train_mean, train_std = x_ref.mean(axis=(0, 1, 2)), x_ref.std(axis=(0, 1, 2))
print("训练集通道均值", train_mean.round(3), "| 标准差", train_std.round(3))
held_acc = float((predict(net, held_u8) == held_y).mean())
before = 1 - target_error
after = float((predict(net, match_channels(target_u8, train_mean, train_std)) == held_y).mean())
print(f"目标开发集（B 实验室：后 5000 张，对比度减半）：校正前 {before:.3f} → 校正后 {after:.3f}（同一批图不加扰动时 {held_acc:.3f}）")
ck.batch_correction(before, after, n_train=N_TRAIN)

print("\n同样的校正用在训练-开发集的其他几种腐蚀上：")
for name, s in [("contrast", 3), ("brightness", 5), ("gaussian_noise", 3), ("gaussian_blur", 5)]:
    fixed = float((predict(net, match_channels(CORRUPTIONS[name](train_dev_u8, s), train_mean, train_std)) == train_dev_y).mean())
    print(f"{name:15s} 严重度 {s}：{sweep[name][s - 1]:.3f} → {fixed:.3f}")

作者的一次运行（CPU，种子 42）：B 实验室的图 0.383 → 0.485，不加扰动时是 0.554——不用重训，就收回了一半以上的错配（这是目标开发集上的数；真实项目里要在另留的目标测试集上再确认一次）。用在其他腐蚀上：对比度严重度 3 从 0.305 回到 0.442、亮度严重度 5 从 0.460 回到 0.533；高斯噪声几乎不变，模糊只回升约 2 个百分点。

规律很清楚：**整体强度、对比度这类偏移能校正一大半，噪声、模糊这类丢了信息的偏移校正不回来**——标准化只能平移和缩放数值，补不回被抹掉的细节。生物数据也一样：染色深浅、曝光强弱的批次差异可以按批次标准化，测序深度不够、离焦这类信息损失就不行。

批次校正之外还有两条出路：

1. **让训练数据更像目标数据**：把目标分布的变化做成数据增强（前面刚讲过"增强也是先验"）。作者试过训练时随机降低对比度（c 在 0.3–1 之间均匀抽，一半的图做，其余配置不变，CPU 重训约 16 秒）：对比度减半的图上准确率到 0.504，可干净图掉了 2.5 个百分点——没有免费的午餐。有空可以自己试试。注意这是在适配一个**已知的**目标偏移；如果是做鲁棒性评测，原作者明确反对在测试用的腐蚀类型上训练（那会高估模型的鲁棒性）。
2. **收集目标分布的数据**：哪怕只有几百张带标签的 B 实验室图，也能拿来做目标开发集、做微调。

**延伸阅读**：Recht et al.（[2018](https://arxiv.org/abs/1806.00451)）按 CIFAR-10 原来的流程重新收集了一份测试集 CIFAR-10.1，几十个模型的准确率都掉了 4–10 个百分点——即使"同一种"数据，重新采集一次也会带来分布偏移。它的图片来自 Tiny Images 数据集，后者已在 2020 年被作者撤回，并要求不再使用。CIFAR-10 本身也是从 Tiny Images 里挑出来、人工核对过标签的小子集，至今仍由多伦多大学发布，所以课程继续使用；CIFAR-10.1 的许可写明图片"按 Tiny Images 的方式使用"，在后者已撤回的情况下我们不再下载它。

---
## S3 · 卷积搬到文本上：TextCNN

文本也有"局部性"：*waste of time*、*not good* 这类短语的意思由**相邻**的几个词决定，而且同一个短语出现在影评开头还是结尾，意思都一样（平移等变性）。把一条影评看成一串词向量，用**一维卷积**沿着词的方向滑动——这就是 Kim (2014) 的 **TextCNN**：

```
词 id (N, L) → Embedding → (N, L, C) → 转置 → (N, C, L)
     → 宽度 3 / 4 / 5 的 Conv1d 各 64 个核 → ReLU → 沿词的方向取最大值（max-over-time pooling）
     → 拼接 (N, 192) → Dropout 0.5 → Linear → (N, 2)
```

### 把影评变成整数矩阵

分词、建词表不是本课的内容，工具包提供现成的 `text.imdb_arrays()`：把每条影评切成词、映射为整数 id，截断或补齐到 256 个词；词表（2 万词）**只用训练集建立**——第 6 周讲过的防泄漏原则。id 0 是填充符 `<pad>`，id 1 是词表外的词 `<unk>`。

In [ ]:
Xtr, ytr, Xte, yte, vocab = text.imdb_arrays(max_len=256, max_vocab=20000)
itos = {i: w for w, i in vocab.items()}           # id → 词，分析时用
print("训练:", Xtr.shape, Xtr.dtype, "  测试:", Xte.shape, "  词表大小:", len(vocab))
print("第一条影评的前 12 个 id:", Xtr[0, :12])
print("还原成词:", " ".join(itos[i] for i in Xtr[0, :12]))

### 本周头号 bug：Embedding 与 Conv1d 的维度约定不同

`nn.Embedding` 是一张可学习的**查找表**：每个词 id 换成一个 64 维的向量。打开官方文档，读 **Shape** 一节：

- [nn.Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html)：输入 "(\*)"，输出 "(\*, H), where \* is the input shape and H = embedding_dim"——一个 batch 的词 id `(N, L)` 变成 `(N, L, C)`；
- [nn.Conv1d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv1d.html)：输入 "(N, C_in, L_in) or (C_in, L_in)"——**通道维必须在长度维前面**。

（`padding_idx=0` 让填充位置的向量恒为 0、也不接受梯度更新。）直接把 Embedding 的输出喂给 Conv1d 会怎样？

In [ ]:
emb = nn.Embedding(len(vocab), 64, padding_idx=0)
conv = nn.Conv1d(in_channels=64, out_channels=64, kernel_size=3)
xb = torch.tensor(Xtr[:2])                        # (2, 256)：两条影评
e = emb(xb)
print("Embedding 输出:", tuple(e.shape), "  ← (N, L, C)")

In [ ]:
# 读这个报错的最后一行：Conv1d 把第 1 维（长度 256）当成了通道数
conv(e)

关键句：`expected input[2, 256, 64] to have 64 channels, but got 256 channels instead`。修复只要一行——交换第 1、2 维：

In [ ]:
e_t = e.transpose(1, 2)                           # (N, L, C) → (N, C, L)
print("转置后:", tuple(e_t.shape), "→ Conv1d 输出:", tuple(conv(e_t).shape), "  ← 256 − 3 + 1 = 254 个位置")

两个提醒：

- **不要用 `reshape` / `view` 去"凑"形状**：形状能凑对，数值却被打乱——每个词的向量会被拆散（练习 E4 让你亲眼看到）；
- 更阴险的情形：如果 embedding 维度**恰好等于**序列长度，不转置也**不会报错**——模型照样训练，只是卷积在"词向量的维度"上滑动，学到的东西毫无意义。所以每一行都注释 shape。

### TextCNN（Kim 2014）

每种宽度一个 `Conv1d`；卷积后沿词的方向取最大值（`max(dim=2)`），每个核只留下它在整条影评里**最强**的那一次激活；三种宽度的结果拼起来，过 dropout 和全连接层：

In [ ]:
class TextCNN(nn.Module):
    """Kim (2014)：多宽度 Conv1d + max-over-time pooling + dropout"""

    def __init__(self, vocab_size, emb_dim=64, n_filters=64, kernel_sizes=(3, 4, 5),
                 n_classes=2, dropout=0.5):
        super().__init__()
        # 在这里写你的代码
        raise NotImplementedError

    def forward(self, x):                                      # x: (N, L) 的词 id
        # 在这里写你的代码
        raise NotImplementedError


ck.textcnn_shapes(TextCNN)
print(f"参数量 {n_params(TextCNN(len(vocab))):,}，其中 Embedding 表 {len(vocab) * 64:,}")

注意参数的分布：133 万个参数里，128 万个在 Embedding 表上——模型的大部分"知识"，要靠 5,000 条影评从随机数开始学出每个词的向量。记住这一点，它决定了下面的结果。

### 训练 5 个 epoch

训练循环与 MNIST 完全相同（`train_one_epoch` 原样复用）；`TensorDataset` 把两个数组打包成 Dataset。CPU 约 1 分钟，GPU 几秒钟。**测试集只在训练结束后看一次**——不按测试集挑 epoch。

In [ ]:
imdb_loader = DataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(ytr)), batch_size=64, shuffle=True)
imdb_test_loader = DataLoader(TensorDataset(torch.tensor(Xte), torch.tensor(yte)), batch_size=256)

experiments.set_seed(42)
textcnn = TextCNN(len(vocab)).to(device)
optimizer = torch.optim.Adam(textcnn.parameters(), lr=1e-3)
t0 = time.time()
for epoch in range(5):
    loss = train_one_epoch(textcnn, imdb_loader, nn.CrossEntropyLoss(), optimizer, device)
    print(f"epoch {epoch + 1}: train loss {loss:.4f}")
textcnn_acc = evaluate(textcnn, imdb_test_loader, device)
print(f"TextCNN 测试准确率 {textcnn_acc:.4f}   用时 {time.time() - t0:.0f}s")
ck.textcnn_accuracy(textcnn_acc)

和第 6 周的基线放在一起——同一份训练/测试划分，TF-IDF + 逻辑回归几秒钟就能重跑：

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

train_df, test_df = data.load_imdb("train"), data.load_imdb("test")
tfidf = TfidfVectorizer(min_df=2)                       # 第 6 周的配置，只在训练集上 fit
clf = LogisticRegression(max_iter=1000).fit(tfidf.fit_transform(train_df["text"]), train_df["label"])
tfidf_acc = clf.score(tfidf.transform(test_df["text"]), test_df["label"])
print(f"TF-IDF + LR（第 6 周基线）: {tfidf_acc:.4f}   TextCNN: {textcnn_acc:.4f}")
ck.tfidf_baseline(tfidf_acc)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
names = ["Majority class", "TF-IDF + LR", "TextCNN (5 epochs)"]
bars = ax.bar(names, [0.5, tfidf_acc, textcnn_acc], color=["gray", "C0", "C2"])
ax.bar_label(bars, fmt="%.3f")
ax.set_ylim(0.4, 1.0)
ax.set_ylabel("Test accuracy")
ax.set_title("On 5k reviews, a from-scratch TextCNN trails TF-IDF + LR")
plt.show()

### 诚实的比较：深度模型没有赢

TextCNN 比 TF-IDF 基线**低了约 10 个百分点**。这不是代码写错了——这是本周最值得记住的真实现象。三段事实：

1. **从零学 embedding、5 个 epoch**：约 0.75–0.77（就是你刚跑出的数）；
2. **训满 15 个 epoch**：约 0.81–0.82（作者实测；你可以把上面的 `range(5)` 改成 15 自己试）——**还是没赢**；
3. 文献里 TextCNN 在 IMDB 上的 85–89%（例如《动手学深度学习》[D2L 16.3 节](https://d2l.ai/chapter_natural-language-processing-applications/sentiment-analysis-cnn.html)（旧版编号 15.3）报告测试准确率 0.868），用的是**全量 25,000 条**训练影评 + **预训练词向量**（GloVe）。我们只有 5,000 条，而且每个词的向量从随机数开始学。

回到上面的参数分布：128 万个 embedding 参数要靠 5,000 条影评去学；而 TF-IDF + 逻辑回归只需为每个词学**一个**权重，词的重要性还由 IDF 直接算好了。

> **架构是先验，但数据量是先验的燃料。** "相邻的词组成短语"这条假设没有错，可是在小数据上，它抵不过一个更简单、参数更少的强基线。第 13–14 周你会看到，**预训练**——先在海量文本上把词的表示学好——正是给先验加满燃料的办法。

这也是第 9 周"强基线"一课的回响：如果没有那条 0.8665 的基线，你很可能会觉得 0.77 是个不错的数，还会写进报告里。

### 卷积核 ≈ 可学习的 k-gram 检测器

宽度为 3 的核每次看 3 个相邻的词——它就是一个**可学习的三元组（trigram）检测器**：与它的权重"对得上"的三个词，激活值最大。max-over-time pooling 只保留每条影评里**最强**的那一处，所以每个核问的问题是："这条影评里**有没有**出现我在找的那种短语？"——不管它出现在开头还是结尾。

验证一下。先从全连接层的权重里挑出最"偏好评"和最"偏差评"的核（`fc.weight[1] − fc.weight[0]` 最大 / 最小的那个），再遍历测试集，找出让它激活最强的几个 k-gram：

In [ ]:
Xte_t = torch.tensor(Xte)


def top_kgrams(model, conv_idx, f, n=5):
    """第 conv_idx 个卷积层里第 f 个核：在测试集上激活最强的 n 个不同 k-gram"""
    conv = model.convs[conv_idx]
    k = conv.kernel_size[0]
    best, where = [], []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(Xte_t), 500):
            # 在这里写你的代码
            raise NotImplementedError
            best.append(m.cpu())
            where.append(pos.cpu())
    best, where = torch.cat(best), torch.cat(where)
    found = []
    for r in best.argsort(descending=True).tolist():           # 从最强往下找，跳过重复的短语
        gram = " ".join(itos[int(t)] for t in Xte[r, where[r]:where[r] + k])
        if gram not in found:
            found.append(gram)
        if len(found) == n:
            break
    return found


W = textcnn.fc.weight.detach().cpu()                  # (2, 192)：每个核对"差评 / 好评"两个 logit 的权重
pos_pref = W[1] - W[0]                                # > 0：这个核激活越强，越把预测推向好评
for conv_idx, k in enumerate((3, 4, 5)):
    s = pos_pref[conv_idx * 64:(conv_idx + 1) * 64]
    for label, f in [("最偏好评", int(s.argmax())), ("最偏差评", int(s.argmin()))]:
        print(f"宽度 {k} · {label}的核 #{f}: {top_kgrams(textcnn, conv_idx, f)}")

具体短语会随种子和设备不同（GPU 与 CPU 上训练出的核不一样），但通常能看到：偏差评的核在找 *worst movie ever*、*waste of time*、*waste your money*、*too bad* 一类短语，偏好评的核在找 *great*、*excellent*、*best* 附近的组合。也有一些核找到的短语看不出情感（比如 *as well as*）——不是每个核都"可解释"。还要留意"误报"：作者的一次运行里，*not too bad* 也点亮了"too bad"检测器——3 个词的窗口看不懂更长距离的否定和转折。这正是 k-gram 的天花板，也是下周**注意力**要解决的问题之一。

### 跨周对比表：从今天开始积累

第 6 周的基线阶梯、今天的 TextCNN、第 11 周的 Transformer、第 13–14 周的预训练模型——全都在**同一份** IMDB 子集上。把它们放进一张表，每周追加一行：

In [ ]:
board = pd.DataFrame([
    {"model": "Majority class", "test_acc": 0.5000, "week": 6},
    {"model": "BoW + LR", "test_acc": 0.8550, "week": 6},
    {"model": "TF-IDF + LR", "test_acc": round(tfidf_acc, 4), "week": 6},
    {"model": "TextCNN (from scratch, 5 epochs)", "test_acc": round(textcnn_acc, 4), "week": 10},
])
print(board.to_string(index=False))
ck.imdb_board(board)

os.makedirs("outputs", exist_ok=True)
board.to_csv("outputs/imdb_results.csv", index=False)
print("已保存 outputs/imdb_results.csv")

**保存好这个 CSV**：Colab 里的文件会随会话结束而消失——下载到本地，或存进 Google Drive。第 11、13、14 周会读入它并追加新行（列名固定为 `model`、`test_acc`、`week`）。

---
## S4 · RNN 概念课 + 研究方法微课：消融实验

### 另一种处理序列的思路：循环神经网络（RNN）

TextCNN 只看固定宽度的窗口。**RNN**（循环神经网络）换了一个思路：像人读书一样从左到右逐词读，边读边更新一个**隐藏状态** $h_t$——对"到目前为止读到了什么"的一份摘要：

$$h_t = \tanh\left(W_h\, h_{t-1} + W_x\, x_t + b\right)$$

每一步用的都是**同一组**权重 $W_h, W_x$——和卷积核在不同位置共享权重是同一个思想，只不过共享发生在"时间"上。把循环按时间**展开**，它就是一个很深的网络：

```
  x_1          x_2          x_3                    x_T
   │            │            │                      │
   ▼            ▼            ▼                      ▼
h_0 ─▶ [RNN] ─h_1─▶ [RNN] ─h_2─▶ [RNN] ─h_3─▶ … ─▶ [RNN] ─h_T─▶ 线性层 → 好评 / 差评
```

读完最后一个词，$h_T$ 就是整条影评的表示，接一个线性层做分类。

### 梯度消失：链式法则乘出来的麻烦

反向传播时，损失要把梯度一路传回第 1 个词，得穿过 $T$ 个时间步。第 6–7 周的链式法则告诉我们，这是 $T$ 个局部导数的**连乘**。假设每一步的局部导数大约都是同一个数 $a$：

In [ ]:
for a in [0.5, 0.9, 1.0, 1.1]:
    print(f"a = {a}:   a^10 = {a ** 10:.3g}   a^50 = {a ** 50:.3g}   a^256 = {a ** 256:.3g}")

T = np.arange(1, 101)
fig, ax = plt.subplots(figsize=(6, 3))
for a in [0.5, 0.9, 1.0, 1.1]:
    ax.semilogy(T, a ** T, label=f"a = {a}")
ax.set_xlabel("Time steps the gradient travels back")
ax.set_ylabel("Gradient scale (log)")
ax.set_title("Repeated multiplication makes gradients vanish or explode")
ax.legend()
plt.show()

$0.9^{50} \approx 0.005$：传回 50 个词之前，梯度只剩千分之五；我们的影评截断后有 256 个词，$0.9^{256} \approx 2 \times 10^{-12}$——开头的词几乎收不到任何学习信号。反过来，$a > 1$ 时梯度**爆炸**（$1.1^{50} \approx 117$）。tanh 的导数 $1 - \tanh^2 \le 1$，所以 RNN 更常遇到的是前者：**梯度消失**——它记不住远处的东西。

### LSTM：给记忆加一条"传送带"

LSTM（Hochreiter & Schmidhuber, 1997；原始版本没有遗忘门，遗忘门由 Gers, Schmidhuber & Cummins 2000 年在 "Learning to Forget" 一文中加入——今天说的 LSTM 一般指带遗忘门的版本）的直观想法：在 $h_t$ 之外，再维护一条**细胞状态** $c_t$，像一条传送带贯穿所有时间步；三个**门**（取值在 0–1 之间的开关，由当前输入和上一步的状态算出来）决定传送带上发生什么：

- **遗忘门** $f_t$：旧记忆保留多少（读到 *but* 时，也许该忘掉前半句的夸奖）；
- **输入门** $i_t$：新信息写入多少；
- **输出门** $o_t$：此刻对外暴露多少记忆。

关键在于传送带的更新是**加法**：

$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$

只要遗忘门接近 1，梯度沿着这条加法路径回传时就不会被反复乘以小于 1 的数，远处的信息得以保留——这和第 11 周 Transformer 里的**残差连接**是同一个思想。

### 为什么最后是注意力胜出

RNN 和 LSTM 统治了序列建模将近二十年，但有两个硬伤：

1. **必须按顺序算**：第 $t$ 步要等第 $t-1$ 步算完，256 个词就得串行 256 步，GPU 的并行能力用不上；
2. **远距离依赖的路太长**：第 1 个词的信息要经过 $T$ 次更新才能影响结尾，一路被稀释。

**注意力**机制让每个词**直接**看序列里的所有其他词：任意两个词之间只隔一步，而且所有位置可以同时计算。代价是它几乎丢掉了"顺序"和"局部性"这些先验——UvA 深度学习课的 [Tutorial 6](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial6/Transformers_and_MHAttention.html)（MIT 许可）专门比较了这种取舍：多头注意力对输入的排列是**等变**的，打乱词序它也察觉不到，位置信息必须另外注入。

| 架构 | 内置的先验（归纳偏置） | 两个远处词之间的路径 | 能否并行 |
| --- | --- | --- | --- |
| MLP | 几乎没有 | — | 能 |
| CNN | 局部性、平移等变性 | 要堆很多层才连得上 | 能 |
| RNN / LSTM | 顺序、权重在时间上共享 | $T$ 步 | 不能 |
| 注意力 | 几乎没有（需显式加位置编码） | 1 步 | 能 |

### 研究方法微课：消融实验——一次只改一处

TextCNN 有好几个设计选择：卷积核宽度、核的数量、max pooling、dropout、embedding 维度……如果有人说"我们的模型效果好"，你要问：**到底是哪个组件起了作用？** 回答这个问题的标准做法叫**消融**（ablation，借自神经科学里"切除一块脑区看功能是否消失"）：从完整模型出发，**每次只拿掉或替换一个组件**，其余全部冻结，看性能变化多少。

| 实验 | 只改了这一处 | 其余 |
| --- | --- | --- |
| 基线 | 核宽 3/4/5、max pooling、dropout 0.5 | — |
| 消融 A | max pooling → mean pooling | 全部同基线 |
| 消融 B | 去掉 dropout | 全部同基线 |
| 消融 C | 只保留宽度为 3 的核 | 全部同基线 |

三条纪律：

1. **一次只改一处**：同时把 pooling 换成 mean、又去掉 dropout，性能掉了 3 个点——你永远说不清是谁的责任；
2. **每一行都要多种子**（≥ 3 个）：第 6 周的最小实验单元原样适用；差异小于种子间的波动，结论就是"差异小于噪声"；
3. **训练预算对齐**：同样的 epoch 数、同样的数据。同时要警惕：固定的预算本身可能偏袒某个设置（比如收敛更快的那个）——这要写进"局限"。

本周作业的**研究实验 2** 就是一次标准的单因素实验：只改卷积核宽度 $k \in \{2, 3, 5, 7\}$，其余全部冻结，3 个种子，与 TF-IDF 基线并列。

### 本周小结

- 卷积 = 共享权重的小窗口滑动；输出尺寸 $\lfloor (W-K+2P)/S \rfloor + 1$
- 两个归纳偏置：**局部性**与**平移等变性**；池化再加一点对小平移的不变性
- MNIST：CNN 用约 1/4 的参数，准确率约 0.975，MLP 约 0.94
- CIFAR-10：照搬官方教程，2 个 epoch 约 55%（对上官方数 = 阳性对照）；MLP 用 6 倍参数只有约 49%
- 数据增强也是先验：告诉模型哪些变换不改变答案
- 误差分析：看 100 个训练-开发集错例、分类计数，收益上限 = 这份数据的误差 × 该类占比；测试集一分为二——前一半是训练-开发集（随便看错例），后一半加扰动是目标开发集（不看错例、不调参数，只算总误差）；真实项目还要从目标分布另留一份测试集，最后报告校正后的成绩
- 训练 / 训练-开发 / 目标开发 / 人类水平四个误差，相邻之差就是可避免偏差、方差与数据错配，先修最大的那块；对比度减半就让 blitz 网络掉了 18 个百分点。整体强度类的偏移像生物里的批次效应，按批次标准化能收回一大半，丢了信息的偏移（噪声、模糊）不行
- TextCNN：Embedding 输出 `(N, L, C)`，Conv1d 要 `(N, C, L)`——**必须 `transpose(1, 2)`**；宽度为 k 的核 ≈ 可学习的 k-gram 检测器
- 诚实的比较：5k 影评上从零训练的 TextCNN（约 0.75–0.77）打不过 TF-IDF + LR（0.8665）——**架构是先验，数据量是先验的燃料**
- RNN：隐藏状态 + 时间展开；连乘导致梯度消失；LSTM 用加法传送带缓解；注意力让任意两个词一步直达
- 消融：一次只改一处、多种子、预算对齐

**预告**：第 11 周 · Attention 与 Transformer——把"按相似度加权平均"写成几行矩阵乘法，从零搭出 Transformer 编码器，让它在同一份 IMDB 上与今天的 TextCNN、TF-IDF 同台比较（对比表再加一行）。